# Contrôle de la couche transports

Entrée : `data/processed/transports.parquet`, produit par `scripts/04_build_transports.py`.

In [1]:
import geopandas as gpd, pandas as pd
t = gpd.read_parquet('../data/processed/transports.parquet')
print(t.crs.to_string(), t.shape)
print('valeurs manquantes :', t[['nom', 'ligne', 'mode', 'geometry']].isna().sum().to_dict())
print('doublons (nom, ligne) :', t.duplicated(['nom', 'ligne']).sum())

EPSG:2154 (1323, 10)
valeurs manquantes : {'nom': 0, 'ligne': 0, 'mode': 0, 'geometry': 0}
doublons (nom, ligne) : 0


## 1. Répartition par mode

In [2]:
t.groupby(['mode', 'existant']).agg(lignes=('ligne', 'nunique'), stations=('nom', 'size'))

,,lignes,stations
mode,existant,,
cable,True,1,5
metro,True,16,405
metro_futur,False,4,65
rer,True,5,247
rer_futur,False,1,10
tram,True,15,281
tram_futur,False,6,70
transilien,True,10,240


In [3]:
t[~t.existant].groupby(['mode', 'ligne', 'statut']).size().to_frame('arrets')

arrets
mode        ligne    statut                   
metro_futur METRO 15 Travaux                36
            METRO 16 Travaux                10
            METRO 17 Travaux                 9
            METRO 18 Travaux                10
rer_futur   RER E    Travaux                10
tram_futur  TRAM 1   AVP                    16
                     Travaux                25
            TRAM 11  Etudes préalables       7
            TRAM 13  Travaux                 4
            TRAM 4   Etudes préalables       1
                     Travaux                 1
            TRAM 7   Travaux                 6
            TRAM 8   AVP                    10

## 2. Emprise et présence en petite couronne

Rattachement approximatif par distance au centre de Paris (Châtelet) : contrôle visuel uniquement, le vrai rattachement se fera aux IRIS à l'étape 4.

In [4]:
print('emprise (WGS84) :', t.to_crs(4326).total_bounds.round(3))
chatelet = gpd.GeoSeries.from_xy([2.347], [48.859], crs=4326).to_crs(2154).iloc[0]
t['km_chatelet'] = (t.distance(chatelet) / 1000).round(1)
t.groupby('mode').km_chatelet.describe(percentiles=[0.5]).round(1)

emprise (WGS84) : [ 1.15  48.181  3.302 49.163]


,count,mean,std,min,50%,max
mode,,,,,,
cable,5.0,15.2,1.3,13.2,15.3,16.3
metro,405.0,3.8,2.3,0.1,3.4,14.6
metro_futur,65.0,12.1,5.0,6.2,10.4,26.0
rer,247.0,18.1,11.8,0.3,16.0,57.6
rer_futur,10.0,33.9,11.0,13.7,33.7,49.4
tram,281.0,10.9,6.0,4.2,9.8,41.0
tram_futur,70.0,11.4,4.5,4.9,10.6,23.7
transilien,240.0,29.8,18.6,2.2,26.6,89.5


## 3. Gares du GPE (lignes 15 à 18)

In [5]:
t[t['mode'] == 'metro_futur'].sort_values(['ligne', 'nom'])[['ligne', 'nom', 'phase', 'statut', 'km_chatelet']]

,ligne,nom,phase,statut,km_chatelet
1200,METRO 15,Arcueil-Cachan,1.0,Travaux,7.0
1224,METRO 15,Bagneux,1.0,Travaux,6.6
1227,METRO 15,Bobigny - Pablo Picasso,1.0,Travaux,9.2
1275,METRO 15,Bois-Colombes,2.0,Travaux,8.3
1198,METRO 15,Bondy,1.0,Travaux,10.5
...,...,...,...,...,...
1183,METRO 18,Massy Palaiseau,1.0,Travaux,16.2
1213,METRO 18,Polytechnique,1.0,Travaux,19.2
1293,METRO 18,Satory,3.0,Travaux,19.5
1181,METRO 18,Université Paris-Saclay,1.0,Travaux,20.7
